In [14]:
import json
import os
from getpass import getpass

import pandas as pd
import requests

# Часть 1. Получение списка репозиториев пользователя GitHub

Используется запрос:

`GET https://api.github.com/users/{username}/repos`

Для публичных репозиториев авторизация не обязательна.


In [15]:
GITHUB_USERNAME = "Kirives"

API_VERSION = "2026-03-10"

print(f"Будут загружены публичные репозитории пользователя: {GITHUB_USERNAME}")

Будут загружены публичные репозитории пользователя: Kirives


### 1.1. Выполнение запроса



In [16]:
url = f"https://api.github.com/users/{GITHUB_USERNAME}/repos"

headers = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": API_VERSION,
}

all_repositories = []
page = 1

while True:
    params = {
        "type": "owner",
        "sort": "full_name",
        "direction": "asc",
        "per_page": 100,
        "page": page,
    }

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=20
    )

    print(f"Страница {page}: HTTP {response.status_code}")
    response.raise_for_status()

    page_data = response.json()

    if not page_data:
        break

    all_repositories.extend(page_data)

    if len(page_data) < 100:
        break

    page += 1

print(f"\nВсего получено репозиториев: {len(all_repositories)}")

Страница 1: HTTP 200

Всего получено репозиториев: 4


### 1.2. Просмотр результата

Выведем несколько основных полей для удобной проверки.

In [17]:
if all_repositories:
    repos_table = pd.DataFrame([
        {
            "id": repo.get("id"),
            "name": repo.get("name"),
            "full_name": repo.get("full_name"),
            "language": repo.get("language"),
            "stars": repo.get("stargazers_count"),
            "forks": repo.get("forks_count"),
            "url": repo.get("html_url"),
        }
        for repo in all_repositories
    ])

    display(repos_table)
else:
    print("У пользователя нет публичных репозиториев.")

,id,name,full_name,language,stars,forks,url
0,1111812221,analyze_2025,Kirives/analyze_2025,Jupyter Notebook,0,0,https://github.com/Kirives/analyze_2025
1,1159717257,clusters_2026,Kirives/clusters_2026,Jupyter Notebook,0,0,https://github.com/Kirives/clusters_2026
2,1209154373,neural_networks_2026,Kirives/neural_networks_2026,Jupyter Notebook,0,0,https://github.com/Kirives/neural_networks_2026
3,1159536601,regression_analyze_2026,Kirives/regression_analyze_2026,Jupyter Notebook,1,0,https://github.com/Kirives/regression_analyze_...


### 1.3. Сохранение JSON-ответа в файл


In [18]:
github_json_filename = f"github_repos_{GITHUB_USERNAME}.json"

with open(github_json_filename, "w", encoding="utf-8") as file:
    json.dump(
        all_repositories,
        file,
        ensure_ascii=False,
        indent=2
    )

print(f"Файл сохранён: {github_json_filename}")
print(f"Размер файла: {os.path.getsize(github_json_filename)} байт")

Файл сохранён: github_repos_Kirives.json
Размер файла: 23508 байт


### 1.4. Проверка сохранённого файла


In [19]:
with open(github_json_filename, "r", encoding="utf-8") as file:
    saved_repositories = json.load(file)

print(f"JSON успешно прочитан обратно.")
print(f"Количество объектов в файле: {len(saved_repositories)}")

if saved_repositories:
    print("\nПример первого репозитория:")
    print(json.dumps(saved_repositories[0], ensure_ascii=False, indent=2)[:2500])

JSON успешно прочитан обратно.
Количество объектов в файле: 4

Пример первого репозитория:
{
  "id": 1111812221,
  "node_id": "R_kgDOQkTofQ",
  "name": "analyze_2025",
  "full_name": "Kirives/analyze_2025",
  "private": false,
  "owner": {
    "login": "Kirives",
    "id": 114109929,
    "node_id": "U_kgDOBs0t6Q",
    "avatar_url": "https://avatars.githubusercontent.com/u/114109929?v=4",
    "gravatar_id": "",
    "url": "https://api.github.com/users/Kirives",
    "html_url": "https://github.com/Kirives",
    "followers_url": "https://api.github.com/users/Kirives/followers",
    "following_url": "https://api.github.com/users/Kirives/following{/other_user}",
    "gists_url": "https://api.github.com/users/Kirives/gists{/gist_id}",
    "starred_url": "https://api.github.com/users/Kirives/starred{/owner}{/repo}",
    "subscriptions_url": "https://api.github.com/users/Kirives/subscriptions",
    "organizations_url": "https://api.github.com/users/Kirives/orgs",
    "repos_url": "https://api.

### 1.5. Скачать JSON из Google Colab



In [20]:
try:
    from google.colab import files
    files.download(github_json_filename)
except ImportError:
    print(f"Не Google Colab. Файл находится в текущей папке: {github_json_filename}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Часть 2. API с авторизацией

Для демонстрации авторизации используется GitHub REST API:

`GET https://api.github.com/user`

Этот endpoint возвращает информацию о пользователе, которому принадлежит переданный токен.


In [21]:
GITHUB_TOKEN = getpass("Вставьте GitHub Personal Access Token: ").strip()

if not GITHUB_TOKEN:
    raise ValueError("Токен не введён. Создайте GitHub Personal Access Token и запустите ячейку ещё раз.")

print("Токен получен и хранится только в оперативной памяти текущей сессии.")

Вставьте GitHub Personal Access Token: ··········
Токен получен и хранится только в оперативной памяти текущей сессии.


### 2.1. Авторизованный запрос


In [22]:
authorized_url = "https://api.github.com/user"

authorized_headers = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "X-GitHub-Api-Version": API_VERSION,
}

authorized_response = requests.get(
    authorized_url,
    headers=authorized_headers,
    timeout=20
)

print(f"HTTP status: {authorized_response.status_code}")

if authorized_response.status_code == 401:
    raise RuntimeError(
        "Ошибка 401: GitHub не принял токен. "
        "Проверьте токен, создайте новый при необходимости и повторите запрос."
    )

authorized_response.raise_for_status()
authorized_user_data = authorized_response.json()

print("Авторизация прошла успешно.")

HTTP status: 200
Авторизация прошла успешно.


### 2.2. Просмотр ответа сервера



In [23]:
fields_to_show = [
    "login",
    "id",
    "name",
    "company",
    "location",
    "public_repos",
    "followers",
    "following",
    "created_at",
    "html_url",
]

result_for_display = {
    field: authorized_user_data.get(field)
    for field in fields_to_show
}

print(json.dumps(result_for_display, ensure_ascii=False, indent=2))

{
  "login": "Kirives",
  "id": 114109929,
  "name": "Kirives",
  "company": null,
  "location": null,
  "public_repos": 4,
  "followers": 0,
  "following": 0,
  "created_at": "2022-09-21T20:11:26Z",
  "html_url": "https://github.com/Kirives"
}


### 2.3. Сохранение ответа авторизованного API в JSON


In [24]:
authorized_json_filename = "github_authorized_user.json"

with open(authorized_json_filename, "w", encoding="utf-8") as file:
    json.dump(
        authorized_user_data,
        file,
        ensure_ascii=False,
        indent=2
    )

print(f"Файл сохранён: {authorized_json_filename}")
print(f"Размер файла: {os.path.getsize(authorized_json_filename)} байт")

Файл сохранён: github_authorized_user.json
Размер файла: 1393 байт


### 2.4. Скачать второй JSON из Google Colab


In [25]:
try:
    from google.colab import files
    files.download(authorized_json_filename)
except ImportError:
    print(f"Не Google Colab. Файл находится в текущей папке: {authorized_json_filename}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>